In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))
from config import CHROMA_DIR, COLLECTION, EMBED_MODEL, DOSSIER_CP, TOP_K
print(CHROMA_DIR.exists())

True


In [4]:
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from src.extraction_cp.extraction import analyser_cp


In [5]:
# La base vectorielle, déjà construite à l'étape 2 : on la rouvre
store = Chroma(collection_name=COLLECTION,
               embedding_function=OllamaEmbeddings(model=EMBED_MODEL),
               persist_directory=str(CHROMA_DIR))
print("Chunks dans la base :", store._collection.count())

Chunks dans la base : 2368


In [6]:
texte_cp, fiche, contrat_id = analyser_cp(DOSSIER_CP / "cp_01_luko_etendue.pdf")
print("Contrat :", contrat_id)

Contrat : luko_hce_2504


In [8]:
question = "Suis-je couvert si on me vole mon vélo dans ma cave ?"
passages = store.similarity_search(question, k=TOP_K, filter={"contrat_id": contrat_id})


In [12]:
print("Extrait de la 1re page gardée :", repr(passages[4].page_content[:250]))

Extrait de la 1re page gardée : "Garantie additionnelle \nNous couvrons aussi les dommages matériels des biens immobiliers du logement assuré, \ndétériorés à l'occasion d'un vol ou d'une tentative de vol. \n\n Conditions de garanties \nÀ la souscription du contrat, vous avez déclaré que "


In [13]:
for i, doc in enumerate(passages, start=1):
    print(f"[{i}] {doc.metadata['fichier']} p.{doc.metadata['page']}")
    print("   ", doc.page_content[:200].replace("\n", " "), "...\n")

[1] luko_habitation-etendue_cg_2504.pdf p.47
    Nous vous garantissons également contre les recours dirigés contre vous en  raison des dommages corporels subis par vos employés de maison en cas  d'accident du travail (ou de maladie professionnelle  ...

[2] luko_habitation-etendue_cg_2504.pdf p.12
    3. Quels sont les biens assurés ? ...

[3] luko_habitation-etendue_cg_2504.pdf p.23
    1.5. Le vol et le vandalisme   Vol  Nous garantissons, sous réserve du niveau de protection approprié, le vol de vos biens commis à  l'intérieur du logement assuré survenu dans l'une des circonstances ...

[4] luko_habitation-etendue_cg_2504.pdf p.36
    Garanties supplémentaires  Nous couvrons également :  ● le vol des biens mobiliers (à l'exclusion des objets de valeurs) et des installations fixes  telles que le système de filtration, situés dans le ...

[5] luko_habitation-etendue_cg_2504.pdf p.24
    Garantie additionnelle  Nous couvrons aussi les dommages matériels des biens immobiliers du logeme